# Imports 

In [3]:
import pandas as pd 
import numpy as np 
from sentence_transformers import SentenceTransformer
from scipy import stats
import mlflow

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# baseline query distribution

In [4]:
baseline_queries = [
    "credit card fraud unauthorized charges",
    "mortgage payment late fees",
    "debt collector harassment calls",
    "student loan repayment issues",
    "bank account overdraft fees",
    "credit card billing dispute",
    "mortgage foreclosure notice",
    "debt collection wrong person",
    "student loan servicer problem",
    "bank account unauthorized transfer",
] * 50 

In [5]:
drifted_queries = [
    "crypto wallet unauthorized transfer",
    "bitcoin exchange account frozen",
    "digital asset theft complaint",
    "cryptocurrency fraud exchange",
    "NFT purchase dispute refund",
] * 50 + baseline_queries[:250] 

Embed both query sets

In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6315.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
print("Encoding baseline queries...")
baseline_embeddings = model.encode(baseline_queries, batch_size=64, show_progress_bar=True)

print("Encoding drifted queries...")
drifted_embeddings = model.encode(drifted_queries, batch_size=64, show_progress_bar=True)

print(f"Baseline shape: {baseline_embeddings.shape}")
print(f"Drifted shape: {drifted_embeddings.shape}")

Encoding baseline queries...


Batches: 100%|██████████████████████████████████████████████████████████████████| 8/8 [00:01<00:00,  6.12it/s]


Encoding drifted queries...


Batches: 100%|██████████████████████████████████████████████████████████████████| 8/8 [00:01<00:00,  5.44it/s]


Baseline shape: (500, 384)
Drifted shape: (500, 384)


# Reduce embedding to 1d 

In [11]:
def detect_embedding_drift(baseline_emb,production_emb,threshold=0.05):
    n_dims = baseline_emb.shape[1]
    drifted_dims = 0
    ks_stats = []

    for dim in range(n_dims):
        stat, p_value = stats.ks_2samp(
            baseline_emb[:,dim],
            production_emb[:,dim]
        )
        ks_stats.append((stat,p_value))
        if p_value < threshold:
            drifted_dims += 1

    drifted_ratio = drifted_dims/ n_dims
    avg_ks = np.mean([s for s, _ in ks_stats])

    return {
        "drifted_dimensions": drifted_dims,
        "total_dimensions": n_dims,
        "drift_ratio": round(drifted_ratio, 4),
        "avg_ks_statistic": round(avg_ks, 4),
        "drift_detected": drifted_ratio > 0.10  # alert if >10% dims drifted
    }

In [14]:
print("=== Scenario 1: Baseline vs Baseline (no drift) ===")
# Split baseline in half to simulate no drift
half = len(baseline_embeddings) // 2
result_no_drift = detect_embedding_drift(
    baseline_embeddings[:half],
    baseline_embeddings[half:]
)
print(result_no_drift)

print("\n=== Scenario 2: Baseline vs Drifted (crypto queries) ===")
result_drift = detect_embedding_drift(
    baseline_embeddings,
    drifted_embeddings
)
print(result_drift)

=== Scenario 1: Baseline vs Baseline (no drift) ===
{'drifted_dimensions': 0, 'total_dimensions': 384, 'drift_ratio': 0.0, 'avg_ks_statistic': np.float64(0.0), 'drift_detected': False}

=== Scenario 2: Baseline vs Drifted (crypto queries) ===
{'drifted_dimensions': 384, 'total_dimensions': 384, 'drift_ratio': 1.0, 'avg_ks_statistic': np.float64(0.246), 'drift_detected': True}


check similarity score drop

In [15]:
import chromadb

In [16]:
client = chromadb.PersistentClient(path="../data/chroma_db")
collection = client.get_collection("cfpb_complaints")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [17]:
def avg_search_score(queries, model, collection, n=3):
    scores = []
    for query in queries[:20]:  # sample 20 queries
        emb = model.encode([query]).tolist()
        results = collection.query(
            query_embeddings=emb,
            n_results=n,
            include=["distances"]
        )
        top_score = 1 - results["distances"][0][0]
        scores.append(top_score)
    return round(np.mean(scores), 4)

In [18]:
print("\n=== Similarity Score Comparison ===")
baseline_score = avg_search_score(baseline_queries, model, collection)
drifted_score = avg_search_score(drifted_queries, model, collection)
print(f"Baseline avg top-1 score: {baseline_score}")
print(f"Drifted avg top-1 score:  {drifted_score}")
score_drop = baseline_score - drifted_score
print(f"Score drop: {score_drop:.4f}")
if score_drop > 0.05:
    print("🚨 ALERT: Search quality degraded — possible query drift")
else:
    print("✅ Search quality stable")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



=== Similarity Score Comparison ===


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry eve

Baseline avg top-1 score: 0.7485
Drifted avg top-1 score:  0.5379
Score drop: 0.2106
🚨 ALERT: Search quality degraded — possible query drift


In [19]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("semantic-search-cfpb")

/opt/miniconda3/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/15 13:05:36 INFO mlflow.tracking.fluent: Experiment with name 'semantic-search-cfpb' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/spectra/Desktop/ml-ai-journey/Week-8/Project-2-Semantic-Search/notebooks/../mlruns/383920549253588213', creation_time=1776238536852, experiment_id='383920549253588213', last_update_time=1776238536852, lifecycle_stage='active', name='semantic-search-cfpb', tags={}, trace_location=None, workspace='default'>

In [20]:
with mlflow.start_run(run_name="query-drift-detection"):
    mlflow.log_metric("drift_ratio_no_drift", result_no_drift["drift_ratio"])
    mlflow.log_metric("drift_ratio_drifted", result_drift["drift_ratio"])
    mlflow.log_metric("avg_ks_no_drift", result_no_drift["avg_ks_statistic"])
    mlflow.log_metric("avg_ks_drifted", result_drift["avg_ks_statistic"])
    mlflow.log_metric("baseline_search_score", baseline_score)
    mlflow.log_metric("drifted_search_score", drifted_score)
    mlflow.log_metric("score_drop", score_drop)
    print("\nLogged to MLflow")


Logged to MLflow
